<a href="https://colab.research.google.com/github/shreyathespark/Gen-AI-LLM-lab-experiments/blob/main/expt_6_GAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Install packages (Run once if needed)
# !pip install -q sentence-transformers transformers faiss-cpu torch

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import faiss
import numpy as np
import torch

# -------------------------------
# 1. Knowledge Base
# -------------------------------
documents = [
    "The Eiffel Tower is located in Paris, France and was completed in 1889.",
    "Retrieval-Augmented Generation (RAG) combines document retrieval with text generation.",
    "Python is a popular high-level programming language used in AI development.",
    "Vector databases store embeddings and support fast similarity search."
]

# -------------------------------
# 2. Load Embedding Model
# -------------------------------
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# Create embeddings
doc_embeddings = embed_model.encode(
    documents,
    convert_to_numpy=True,
    normalize_embeddings=True
)

# -------------------------------
# 3. Build FAISS Index
# -------------------------------
dimension = doc_embeddings.shape[1]

# Cosine similarity using normalized embeddings
index = faiss.IndexFlatIP(dimension)

index.add(doc_embeddings)

# -------------------------------
# 4. User Query
# -------------------------------
query = "What is RAG in AI?"

query_embedding = embed_model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
)

# Retrieve Top-2 documents
k = 2
distances, indices = index.search(query_embedding, k)

retrieved_chunks = [documents[i] for i in indices[0]]

print("Retrieved Documents:")
for doc in retrieved_chunks:
    print("-", doc)

# -------------------------------
# 5. Build Prompt
# -------------------------------
context = "\n".join(retrieved_chunks)

prompt = f"""
Answer the question using the context below.

Context:
{context}

Question:
{query}

Answer:
"""

# -------------------------------
# 6. Load FLAN-T5
# -------------------------------
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# -------------------------------
# 7. Generate Answer
# -------------------------------
inputs = tokenizer(prompt, return_tensors="pt", truncation=True)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=60,
        num_beams=4,
        early_stopping=True
    )

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\nGenerated Answer:")
print(answer)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Retrieved Documents:
- Retrieval-Augmented Generation (RAG) combines document retrieval with text generation.
- Python is a popular high-level programming language used in AI development.


tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]


Generated Answer:
combines document retrieval with text generation
